In [ ]:
from osgeo import gdal
import numpy as np
def readtif(fileName):
    dataset = gdal.Open(fileName)
    width = dataset.RasterXSize
    height = dataset.RasterYSize
    GdalImg_data = dataset.ReadAsArray(0, 0, width, height)
    return GdalImg_data

In [ ]:
def dataPreprocess(img, label, classNum, colorDict_GRAY ):
    img = img / img.max()
    for i in range(colorDict_GRAY.shape[0]):
        label[label == colorDict_GRAY[i][0]] = i
    new_label = np.zeros(label.shape + (classNum,))
    for i in range(classNum):
        new_label[label == i,i] = 1                                          
    label = new_label
    return (img, label)

In [ ]:
import cv2
def color_dict(labelFolder, classNum):
    colorDict = []
    ImageNameList = os.listdir(labelFolder)
    for i in range(len(ImageNameList)):
        ImagePath = labelFolder + "/" + ImageNameList[i]
        img = cv2.imread(ImagePath).astype(np.uint32)
        if(len(img.shape) == 2):
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB).astype(np.uint32)
        img_new = img[:,:,0] * 1000000 + img[:,:,1] * 1000 + img[:,:,2]
        unique = np.unique(img_new)
        for j in range(unique.shape[0]):
            colorDict.append(unique[j])
        colorDict = sorted(set(colorDict))
        if(len(colorDict) == classNum):
            break
    colorDict_RGB = []
    for k in range(len(colorDict)):
        color = str(colorDict[k]).rjust(9, '0')
        color_RGB = [int(color[0 : 3]), int(color[3 : 6]), int(color[6 : 9])]
        colorDict_RGB.append(color_RGB)
    colorDict_RGB = np.array(colorDict_RGB)
    colorDict_GRAY = colorDict_RGB.reshape((colorDict_RGB.shape[0], 1 ,colorDict_RGB.shape[1])).astype(np.uint8)
    colorDict_GRAY = cv2.cvtColor(colorDict_GRAY, cv2.COLOR_BGR2GRAY)
    return colorDict_RGB, colorDict_GRAY

In [ ]:
import cv2
import random
def trainGenerator(batch_size, train_image_path, train_label_path, classNum, colorDict_GRAY, resize_shape = None):
    imageList = os.listdir(train_image_path)
    labelList = os.listdir(train_label_path)
    img = readtif(train_image_path + "\\" + imageList[0])
    img = img.swapaxes(1, 0)
    img = img.swapaxes(1, 2)
    while(True):
        img_generator = np.zeros((batch_size, img.shape[0], img.shape[1], img.shape[2]), np.uint8)
        label_generator = np.zeros((batch_size, img.shape[0], img.shape[1]), np.uint8)
        if(resize_shape != None):
            img_generator = np.zeros((batch_size, resize_shape[0], resize_shape[1], resize_shape[2]), np.uint8)
            label_generator = np.zeros((batch_size, resize_shape[0], resize_shape[1]), np.uint8)
        rand = random.randint(0, len(imageList) - batch_size)
        for j in range(batch_size):
            img = readtif(train_image_path + "\\" + imageList[rand + j])
            img = img.swapaxes(1, 0)
            img = img.swapaxes(1, 2)
            if(resize_shape != None):
                img = cv2.resize(img, (resize_shape[0], resize_shape[1]))
            img_generator[j] = img
            label = readtif(train_label_path + "\\" + labelList[rand + j]).astype(np.uint8)
            if(len(label.shape) == 3):
                label = label.swapaxes(1, 0)
                label = label.swapaxes(1, 2)
                label = cv2.cvtColor(label, cv2.COLOR_RGB2GRAY)
            if(resize_shape != None):
                label = cv2.resize(label, (resize_shape[0], resize_shape[1]))
            label_generator[j] = label
        img_generator, label_generator = dataPreprocess(img_generator, label_generator, classNum, colorDict_GRAY)
        yield (img_generator,label_generator)

In [ ]:
import keras
from keras.models import *
from keras.layers import *
from keras import layers
import keras.backend as K
IMAGE_ORDERING = 'channels_last'
def CBAM_block(cbam_feature,ratio=8):
    cbam_feature = channel_attention(cbam_feature, ratio)
    cbam_feature = spatial_attention(cbam_feature)
    return cbam_feature
def channel_attention(input_feature,ratio):
    channel = input_feature.shape[-1]
    shared_layer_one = Dense(channel // ratio,
                             activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    shared_layer_two = Dense(channel,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)
    max_pool = GlobalMaxPool2D()(input_feature)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)
    cbam = Add()([avg_pool,max_pool])
    cbam = Activation('sigmoid')(cbam)
    return multiply([input_feature,cbam])
def spatial_attention(input_feature):
    avg_pool = Lambda(lambda x:K.mean(x,axis=3,keepdims=True))(input_feature)
    max_pool = Lambda(lambda x:K.max(x,axis=3,keepdims=True))(input_feature)
    concat = Concatenate(axis=3)([avg_pool,max_pool])
    cbam_feature = Conv2D(1,(7,7),strides=1,padding='same',activation='sigmoid')(concat)
    return multiply([input_feature,cbam_feature])

In [ ]:
def focal_loss(gamma=2., alpha=.25):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    def multi_category_focal_loss2_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
        alpha_t = y_true*alpha + (tf.ones_like(y_true)-y_true)*(1-alpha)
        y_t = tf.multiply(y_true, y_pred) + tf.multiply(1-y_true, 1-y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(tf.subtract(1., y_t), gamma)
        fl = tf.multiply(tf.multiply(weight, ce), alpha_t)
        loss = tf.reduce_mean(fl)
        return loss
    return multi_category_focal_loss2_fixed
smooth = 1. 
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true) 
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f * y_true_f) + K.sum(y_pred_f * y_pred_f) + smooth)
def dice_coef_loss(y_true, y_pred):
    return 1. - dice_coef(y_true, y_pred)


In [ ]:
from tensorflow.keras.models import Model
from keras.layers import *
#from tensorflow.keras.layers import Input, concatenate, Conv2D, MaxPooling2D, Dropout,Conv2DTranspose, Activation
#from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
def unet(num_classes, input_shape, lr_init):
    img_input = Input(input_shape)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(img_input)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
    conv4 = Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = Conv2D(512, 3, activation='relu', padding='same')(conv4)
    drop4 = Dropout(0.5)(conv4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(drop4)
    # Bottleneck
    conv5 = Conv2D(1024, 3, activation='relu', padding='same')(pool4)
    CBAM = CBAM_block(conv5)
    conv5 = Conv2D(1024, 3, activation='relu', padding='same')(CBAM)
    drop5 = Dropout(0.5)(conv5)
    # Decoder (expansive path)
    up6 = Conv2D(512, 2, activation='relu', padding='same')(UpSampling2D(size=(2, 2))(drop5))
    merge6 = concatenate([drop4, up6], axis=3)
    conv6 = Conv2D(512, 3, activation='relu', padding='same')(merge6)
    conv6 = Conv2D(512, 3, activation='relu', padding='same')(conv6) 
    CBAM1 = CBAM_block(conv6)
    up7 = Conv2D(256, 2, activation='relu', padding='same')(UpSampling2D(size=(2, 2))(CBAM1))
    merge7 = concatenate([conv3, up7], axis=3)
    conv7 = Conv2D(256, 3, activation='relu', padding='same')(merge7)
    conv7 = Conv2D(256, 3, activation='relu', padding='same')(conv7)
    CBAM2 = CBAM_block(conv7)
    up8 = Conv2D(128, 2, activation='relu', padding='same')(UpSampling2D(size=(2, 2))(CBAM2))
    merge8 = concatenate([conv2, up8], axis=3)
    conv8 = Conv2D(128, 3, activation='relu', padding='same')(merge8)
    conv8 = Conv2D(128, 3, activation='relu', padding='same')(conv8)
    CBAM3 = CBAM_block(conv8)
    up9 = Conv2D(64, 2, activation='relu', padding='same')(UpSampling2D(size=(2, 2))(CBAM3))
    merge9 = concatenate([conv1, up9], axis=3)
    conv9 = Conv2D(64, 3, activation='relu', padding='same')(merge9)
    CBAM4= CBAM_block(conv9)
    conv9 = Conv2D(64, 3, activation='relu', padding='same')(CBAM4)
    conv9 = Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
    conv9 = Conv2D(num_classes, 3, activation='softmax', padding='same')(conv9)
    model = Model(img_input, conv9)
    model.compile(optimizer=Adam(learning_rate=lr_init),
                  loss=[dice_coef_loss],
                  metrics=["accuracy"])
    return model

In [ ]:
import os
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
from tensorflow import keras
train_image_path =your_path
train_label_path = your_path
validation_image_path = your_path
validation_label_path = your_path
model_path =your_path
batch_size = 4
classNum = 3
input_size = (512, 512, 5)
epochs = 200
learning_rate = 1e-4
premodel_path =None
model_path =r"E:\yx\bm\train_and_results\train_final\output\model\unet_cbam_fin111.hdf5"
train_num = len(os.listdir(train_image_path))
validation_num = len(os.listdir(validation_image_path))
steps_per_epoch = train_num / batch_size
validation_steps = validation_num / batch_size
colorDict_RGB, colorDict_GRAY = color_dict(train_label_path, classNum)
train_Generator = trainGenerator(batch_size,
                                 train_image_path, 
                                 train_label_path,
                                 classNum ,
                                 colorDict_GRAY,
                                 input_size)
validation_data = trainGenerator(batch_size,
                                 validation_image_path,
                                 validation_label_path,
                                 classNum,
                                 colorDict_GRAY,
                                 input_size)
model  =  unet(num_classes =classNum , input_shape = input_size, lr_init =learning_rate)
model_checkpoint = ModelCheckpoint(model_path,
                                   monitor = 'loss',
                                   verbose = 1,
                                   save_best_only = True)


In [ ]:
history = model.fit(train_Generator,
                    steps_per_epoch = steps_per_epoch,
                    epochs = epochs,
                    callbacks = [model_checkpoint],
                    validation_data = validation_data,
                    validation_steps = validation_steps)

In [ ]:
def testGenerator(test_iamge_path, resize_shape = None):
    imageList = os.listdir(test_iamge_path)
    for i in range(len(imageList)):
        img = readtif(test_iamge_path + "\\" + imageList[i])
        img = img.swapaxes(1, 0)
        img = img.swapaxes(1, 2)
        img = img / img.max()
        if(resize_shape != None):
            img = cv2.resize(img, (resize_shape[0], resize_shape[1]))
        img = np.reshape(img, (1, ) + img.shape)
        yield img

In [ ]:
def saveResult(test_image_path, test_predict_path, model_predict, color_dict, output_size):
    imageList = os.listdir(test_image_path)
    for i, img in enumerate(model_predict):
        channel_max = np.argmax(img, axis = -1)
        img_out = np.uint8(color_dict[channel_max.astype(np.uint8)])
        img_out = cv2.resize(img_out, (output_size[0], output_size[1]), interpolation = cv2.INTER_NEAREST)
        cv2.imwrite(test_predict_path + "\\" + imageList[i][:-4] + ".png", img_out)

In [ ]:
from tensorflow.keras.models import load_model
new_model = your_path
model = load_model(new_model,custom_objects={'dice_coef_loss': dice_coef_loss,'dice_coef':dice_coef})
#model = load_model(new_model,custom_objects={'multi_category_focal_loss2_fixed': focal_loss})

In [15]:
predict = model.predict(testGenerator(predict_root))
output_size = (512,512,3)
result = saveResult(predict_root,output_root,predict,colorDict_RGB,output_size)

In [ ]:
from osgeo import gdal
import numpy as np
from keras.models import load_model
from keras import losses
import datetime
import math
import sys
def readTif(fileName, xoff = 0, yoff = 0, data_width = 0, data_height = 0):
    dataset = gdal.Open(fileName)
    width = dataset.RasterXSize 
    height = dataset.RasterYSize 
    bands = dataset.RasterCount 
    if(data_width == 0 and data_height == 0):
        data_width = width
        data_height = height
    data = dataset.ReadAsArray(xoff, yoff, data_width, data_height)
    geotrans = dataset.GetGeoTransform()
    proj = dataset.GetProjection()
    return width, height, bands, data, geotrans, proj
def writeTiff(im_data, im_geotrans, im_proj, path):
    if 'int8' in im_data.dtype.name:
        datatype = gdal.GDT_Byte
    elif 'int16' in im_data.dtype.name:
        datatype = gdal.GDT_UInt16
    else:
        datatype = gdal.GDT_Float32
    if len(im_data.shape) == 3:
        im_bands, im_height, im_width = im_data.shape
    elif len(im_data.shape) == 2:
        im_data = np.array([im_data])
        im_bands, im_height, im_width = im_data.shape
    driver = gdal.GetDriverByName("GTiff")
    dataset = driver.Create(path, int(im_width), int(im_height), int(im_bands), datatype)
    if(dataset!= None):
        dataset.SetGeoTransform(im_geotrans) 
        dataset.SetProjection(im_proj) 
    for i in range(im_bands):
        dataset.GetRasterBand(i+1).WriteArray(im_data[i])
    del dataset
def TifCroppingArray(img, SideLength):
    TifArrayReturn = []
    ColumnNum = int((img.shape[0] - SideLength * 2) / (512 - SideLength * 2))
    RowNum = int((img.shape[1] - SideLength * 2) / (512 - SideLength * 2))
    for i in range(ColumnNum):
        TifArray = []
        for j in range(RowNum):
            cropped = img[i * (512 - SideLength * 2) : i * (512 - SideLength * 2) + 512,
                          j * (512 - SideLength * 2) : j * (512 - SideLength * 2) + 512]
            TifArray.append(cropped)
        TifArrayReturn.append(TifArray)
    for i in range(ColumnNum):
        cropped = img[i * (512 - SideLength * 2) : i * (512 - SideLength * 2) + 512,
                      (img.shape[1] - 512) : img.shape[1]]
        TifArrayReturn[i].append(cropped)
    TifArray = []
    for j in range(RowNum):
        cropped = img[(img.shape[0] - 512) : img.shape[0],
                      j * (512-SideLength*2) : j * (512 - SideLength * 2) + 512]
        TifArray.append(cropped)
    cropped = img[(img.shape[0] - 512) : img.shape[0],
                  (img.shape[1] - 512) : img.shape[1]]
    TifArray.append(cropped)
    TifArrayReturn.append(TifArray)
    ColumnOver = (img.shape[0] - SideLength * 2) % (512 - SideLength * 2) + SideLength
    RowOver = (img.shape[1] - SideLength * 2) % (512 - SideLength * 2) + SideLength
    return TifArrayReturn, RowOver, ColumnOver
def labelVisualize(img):
    img_out = np.zeros((img.shape[0],img.shape[1]))
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            img_out[i][j] = np.argmax(img[i][j])
    return img_out
def testGenerator(TifArray):
    for i in range(len(TifArray)):
        for j in range(len(TifArray[0])):
            img = TifArray[i][j]
            img = img / img.max()
            img = np.reshape(img,(1,)+img.shape)
            yield img
def Result(shape, TifArray, npyfile, num_class, RepetitiveLength, RowOver, ColumnOver):
    result = np.zeros(shape, np.uint8)
    j = 0  
    for i,item in enumerate(npyfile):
        img = labelVisualize(item)
        img = img.astype(np.uint8)
        if(i % len(TifArray[0]) == 0):
            if(j == 0):
                result[0 : 512 - RepetitiveLength, 0 : 512-RepetitiveLength] = img[0 : 512 - RepetitiveLength, 0 : 512 - RepetitiveLength]
            elif(j == len(TifArray) - 1):
                result[shape[0] - ColumnOver - RepetitiveLength: shape[0], 0 : 512 - RepetitiveLength] = img[512 - ColumnOver - RepetitiveLength : 512, 0 : 512 - RepetitiveLength]
            else:
                result[j * (512 - 2 * RepetitiveLength) + RepetitiveLength : (j + 1) * (512 - 2 * RepetitiveLength) + RepetitiveLength,
                       0:512-RepetitiveLength] = img[RepetitiveLength : 512 - RepetitiveLength, 0 : 512 - RepetitiveLength]   
        elif(i % len(TifArray[0]) == len(TifArray[0]) - 1):
            if(j == 0):
                result[0 : 512 - RepetitiveLength, shape[1] - RowOver: shape[1]] = img[0 : 512 - RepetitiveLength, 512 -  RowOver: 512]
            elif(j == len(TifArray) - 1):
                result[shape[0] - ColumnOver : shape[0], shape[1] - RowOver : shape[1]] = img[512 - ColumnOver : 512, 512 - RowOver : 512]
            else:
                result[j * (512 - 2 * RepetitiveLength) + RepetitiveLength : (j + 1) * (512 - 2 * RepetitiveLength) + RepetitiveLength,
                       shape[1] - RowOver : shape[1]] = img[RepetitiveLength : 512 - RepetitiveLength, 512 - RowOver : 512]   
            j = j + 1
        else:
            if(j == 0):
                result[0 : 512 - RepetitiveLength,
                       (i - j * len(TifArray[0])) * (512 - 2 * RepetitiveLength) + RepetitiveLength : (i - j * len(TifArray[0]) + 1) * (512 - 2 * RepetitiveLength) + RepetitiveLength
                       ] = img[0 : 512 - RepetitiveLength, RepetitiveLength : 512 - RepetitiveLength]         
            if(j == len(TifArray) - 1):
                result[shape[0] - ColumnOver : shape[0],
                       (i - j * len(TifArray[0])) * (512 - 2 * RepetitiveLength) + RepetitiveLength : (i - j * len(TifArray[0]) + 1) * (512 - 2 * RepetitiveLength) + RepetitiveLength
                       ] = img[512 - ColumnOver : 512, RepetitiveLength : 512 - RepetitiveLength]
            else:
                result[j * (512 - 2 * RepetitiveLength) + RepetitiveLength : (j + 1) * (512 - 2 * RepetitiveLength) + RepetitiveLength,
                       (i - j * len(TifArray[0])) * (512 - 2 * RepetitiveLength) + RepetitiveLength : (i - j * len(TifArray[0]) + 1) * (512 - 2 * RepetitiveLength) + RepetitiveLength,
                       ] = img[RepetitiveLength : 512 - RepetitiveLength, RepetitiveLength : 512 - RepetitiveLength]
    return result


In [ ]:
def pre_img(TifPath,ResultPath,ModelPath):
    area_perc = 0.5
    RepetitiveLength = int((1 - math.sqrt(area_perc)) * 256/ 2)
    testtime = []
    starttime = datetime.datetime.now()
    im_width, im_height, im_bands, im_data, im_geotrans, im_proj = readTif(TifPath)
    im_data = im_data.swapaxes(1, 0)
    im_data = im_data.swapaxes(1, 2)

    TifArray, RowOver, ColumnOver = TifCroppingArray(im_data, RepetitiveLength)
    endtime = datetime.datetime.now()
    model = load_model(ModelPath,custom_objects={'dice_coef_loss': dice_coef_loss,'dice_coef':dice_coef})
    testGene = testGenerator(TifArray)
    results = model.predict(testGene,
                            len(TifArray) * len(TifArray[0]),
                            verbose = 1)
    endtime = datetime.datetime.now()
    result_shape = (im_data.shape[0], im_data.shape[1])
    result_data = Result(result_shape, TifArray, results, 2, RepetitiveLength, RowOver, ColumnOver)
    writeTiff(result_data, im_geotrans, im_proj, ResultPath)
    endtime = datetime.datetime.now()
    time = datetime.datetime.strftime(datetime.datetime.now(), '%Y%m%d-%H%M%S')
    with open('timelog_%s.txt'%time, 'w') as f:
        for i in range(len(testtime)):
            f.write(testtime[i])
            f.write("\r\n")